In [1]:
import numpy as np
import pandas as pd

import torch
from transformers import pipeline

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f" HARDWARE ACCELERATION ACTIVE: {gpu_name} detected.")
    device_id = 0
else:
    print(" WARNING: GPU not detected. PyTorch will use CPU.")
    device_id = -1

# C.Hugging Face Test (This might take a few seconds to download the tiny model the first time)
try:
    sentiment_tester = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device_id) 
    hf_result = sentiment_tester
    print(f"Hugging Face Engine Online: {hf_result}")
except Exception as e:
    print(f" Hugging Face Error: {e}")
print("\n CINEIQ ARCHITECTURE IS FULLY OPERATIONAL")

c:\Users\icehe\anaconda3\envs\cineiq\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: b0ad3968-7b1c-436d-aa4e-2696f1d1c61d)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 1s [Retry 1/5].


 HARDWARE ACCELERATION ACTIVE: NVIDIA GeForce RTX 4050 Laptop GPU detected.


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 8446faa0-dfd0-45f6-9611-a38aaddf2e9d)')' thrown while requesting HEAD https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /distilbert-base-uncased-finetuned-sst-2-english/resolve/main/config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: c9a7ce69-41d2-4b44-8cba-ab3c1c62b8f5)')' thrown while requesting HEAD https://huggingface.co/disti

Hugging Face Engine Online: <transformers.pipelines.text_classification.TextClassificationPipeline object at 0x0000025760F12C20>

 CINEIQ ARCHITECTURE IS FULLY OPERATIONAL


In [ ]:
print("INITIATING BLOCK 1: DATA INGESTION...\n")

# 1. Loading the IMDb Reviews Dataset
# (Make sure the file name matches exactly what you downloaded from Kaggle)
imdb_df = pd.read_csv('IMDB Dataset.csv/IMDB Dataset.csv')

# 2. Inspect the Structure
print(f"Dataset Size: {imdb_df.shape[0]} rows, {imdb_df.shape[1]} columns.\n")

# 3. Check for Missing Values (Crucial before sending text to Hugging Face)
missing_values = imdb_df.isnull().sum()
print("Missing Data Check:")
print(missing_values)

# 4. Preview the first 3 rows
print("\nFirst 3 Reviews:")
imdb_df.head(3)

INITIATING BLOCK 1: DATA INGESTION...

Dataset Size: 50000 rows, 2 columns.

Missing Data Check:
review       0
sentiment    0
dtype: int64

First 3 Reviews:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive


In [4]:
import torch
from transformers import pipeline

print("INITIATING BLOCK 2: SENTIMENT ENGINE (SAMPLE TEST)...\n")

# 1. Grab a tiny sample of 10 reviews
sample_df = imdb_df.head(10).copy()

# 2. Boot up the hardware
device_id = 0 if torch.cuda.is_available() else -1

sentiment_model = pipeline( # type: ignore
    "sentiment-analysis", 
    model="distilbert-base-uncased-finetuned-sst-2-english", 
    device=device_id, 
    truncation=True, 
    max_length=512
)

# 3. Extract the text 
texts = sample_df['review'].tolist()

# 4. Process the text through the GPU silently in batches!
print("Unleashing the GPU silently. Please wait...")
results = sentiment_model(texts, batch_size=16)

# 5. Attach the AI's math back to our dataframe
sample_df['hf_label'] = [r['label'] for r in results]
sample_df['hf_score'] = [r['score'] for r in results]

print("\nProcessing Complete! Let's see how smart the AI is:")
sample_df[['review', 'sentiment', 'hf_label', 'hf_score']]

INITIATING BLOCK 2: SENTIMENT ENGINE (SAMPLE TEST)...



Device set to use cuda:0


Unleashing the GPU silently. Please wait...

Processing Complete! Let's see how smart the AI is:


,review,sentiment,hf_label,hf_score
0,One of the other reviewers has mentioned that ...,positive,NEGATIVE,0.512119
1,A wonderful little production. <br /><br />The...,positive,POSITIVE,0.999377
2,I thought this was a wonderful way to spend ti...,positive,POSITIVE,0.999190
3,Basically there's a family where a little boy ...,negative,NEGATIVE,0.999172
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,POSITIVE,0.998824
5,"Probably my all-time favorite movie, a story o...",positive,POSITIVE,0.999768
6,I sure would like to see a resurrection of a u...,positive,POSITIVE,0.916027
7,"This show was an amazing, fresh & innovative i...",negative,NEGATIVE,0.999637
8,Encouraged by the positive comments about this...,negative,NEGATIVE,0.999722
9,If you like original gut wrenching laughter yo...,positive,POSITIVE,0.999701


In [5]:
import re#regex library for text cleaning
#some dangers!
# 1. The Safe Scenario (Missing Arrows)
# If someone types a stray arrow like "This movie is a 10 < 11", your data is perfectly safe.
# The Regex rule strictly requires both an opening < and a closing >. Because it never finds the closing arrow in that sentence, it ignores it completely and deletes nothing.

# 2. The Danger Scenario (The Regex Trap)
# Here is where your logic is 100% correct, and where this Regex will actually destroy valid data. Imagine a user wrote this review:

# "I loved this movie <3 and the acting was > average!"

# Because of the rule we wrote, the Regex will see the < next to the 3, scan all the way across the sentence until it finds the > next to "average", and delete the entire chunk.
# The review would be corrupted into:

# "I loved this movie average!"

print("CLEANING DATASET NOISE...")

# Use Regex (Regular Expressions) to find anything that looks like an HTML tag and replace it with a space
imdb_df['clean_review'] = imdb_df['review'].str.replace(r'<[^<>]*>', ' ', regex=True)

# Remove extra double spaces that might have been left behind
imdb_df['clean_review'] = imdb_df['clean_review'].str.replace(r'\s+', ' ', regex=True).str.strip()

print("Data cleaning complete! Look at Row 1 now:\n")
print("OLD:", imdb_df['review'].iloc[1][:60])
print("NEW:", imdb_df['clean_review'].iloc[1][:60])

CLEANING DATASET NOISE...
Data cleaning complete! Look at Row 1 now:

OLD: A wonderful little production. <br /><br />The filming techn
NEW: A wonderful little production. The filming technique is very


In [6]:

print("UNLEASHING GPU ON FULL 50K DATASET...\n")

# 1. Extract ALL 50,000 cleaned reviews
full_texts = imdb_df['clean_review'].tolist()

# 2. Process silently in batches 
# (Go grab a snack, this will take about 5 to 10 minutes)
print("Processing 50,000 rows. Please wait...")
full_results = sentiment_model(full_texts, batch_size=16)

# 3. Attach the math back to the main dataframe
imdb_df['hf_label'] = [r['label'] for r in full_results]
imdb_df['hf_score'] = [r['score'] for r in full_results]

# 4. SAVE THE FINAL PIPELINE OUTPUT
# This creates a new file so we don't overwrite your original raw Kaggle data
imdb_df.to_csv('IMDB_processed_sentiment.csv', index=False)

print("\nMASSIVE BATCH COMPLETE!")
print("Saved permanently to: data/IMDB_processed_sentiment.csv")

UNLEASHING GPU ON FULL 50K DATASET...

Processing 50,000 rows. Please wait...

MASSIVE BATCH COMPLETE!
Saved permanently to: data/IMDB_processed_sentiment.csv


In [7]:
# Load a tiny bit of the new file just to verify it saved correctly
check_df = pd.read_csv('IMDB_processed_sentiment.csv')
check_df[['review', 'hf_label', 'hf_score']].head()

,review,hf_label,hf_score
0,One of the other reviewers has mentioned that ...,NEGATIVE,0.825068
1,A wonderful little production. <br /><br />The...,POSITIVE,0.999253
2,I thought this was a wonderful way to spend ti...,POSITIVE,0.999064
3,Basically there's a family where a little boy ...,NEGATIVE,0.994191
4,"Petter Mattei's ""Love in the Time of Money"" is...",POSITIVE,0.999079
